# Week 4 — Market Sizing (TAM/SAM/SOM)

Builds top-down + bottom-up market sizes for all five InGen verticals, reconciles them, and runs a one-at-a-time tornado sensitivity. Market totals are public estimates (which disagree widely, so we use ranges); demand bases come from the Week 3 warehouse.

## 1. Load assumptions and recompute SOMs

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '..')
# market_data.py lives at repo root in the standalone Week 4 tree; fall back to inline if packaged elsewhere
try:
    import market_data as M
except ModuleNotFoundError:
    sys.path.insert(0, '../..'); import market_data as M
rows=[]
for v in M.VERTICALS:
    td=M.top_down(v); b=M.BOTTOM_UP[v]
    som_low=M.bottom_up_som(v,units=b['units_low'],serviceable=b['serviceable_pct_low'],penetration=b['penetration_low'],asp=b['asp_low'])
    som_base=M.bottom_up_som(v)
    som_high=M.bottom_up_som(v,units=b['units_high'],serviceable=b['serviceable_pct_high'],penetration=b['penetration_high'],asp=b['asp_high'])
    rows.append(dict(vertical=v, product=M.ANCHOR_PRODUCT[v], tam_low=td['tam_low'], tam_high=td['tam_high'],
                     sam_low=td['sam_low'], sam_high=td['sam_high'], som_low=som_low, som_base=som_base, som_high=som_high))
df=pd.DataFrame(rows)
fmt=lambda x: f'${x/1e9:.2f}B' if x>=1e9 else f'${x/1e6:.0f}M'
show=df.copy()
for c in ['tam_low','tam_high','sam_low','sam_high','som_low','som_base','som_high']:
    show[c]=show[c].map(fmt)
print(show.to_string(index=False))

## 2. Load the assumptions CSV (the written deliverable)

In [ ]:
a=pd.read_csv('../data/week04/market_sizing_assumptions.csv')
print('rows:', len(a), '| verticals:', a['vertical'].nunique(), '| drivers per vertical:', a.groupby('vertical').size().unique())
print(a.head(8).to_string(index=False))

## 3. Which driver dominates each vertical? (tornado swing)

In [ ]:
sw=a.sort_values(['vertical','swing_abs'],ascending=[True,False])
top=sw.groupby('vertical').first().reset_index()[['vertical','driver','swing_abs']]
top['swing']=top['swing_abs'].map(lambda x: f'${x/1e6:.0f}M')
print('Dominant uncertainty per vertical:')
print(top[['vertical','driver','swing']].to_string(index=False))

## 4. SOM ranges chart

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import numpy as np
fig,ax=plt.subplots(figsize=(9,4.5))
y=np.arange(len(df))
ax.barh(y, (df['som_high']-df['som_low'])/1e6, left=df['som_low']/1e6, color='#8FAADC')
ax.plot(df['som_base']/1e6, y, 'D', color='#C00000', label='base')
ax.set_yticks(y); ax.set_yticklabels(df['product'])
ax.set_xlabel('Bottom-up SOM (USD millions)'); ax.set_title('SOM ranges by vertical (bottom-up)')
ax.legend(); plt.tight_layout(); plt.savefig('week04_som_ranges.png',dpi=120)
print('saved week04_som_ranges.png')

## 5. Takeaways for InGen
- Indoor security (Sentinel Prime AI) has the **largest, most defensible** bottom-up SOM — a real commercial market with guard-replacement economics.
- Eldercare (Fari) SOM is **penetration-limited**: adoption rate is the single biggest swing, so a real pilot conversion number would most tighten the estimate.
- Humanoid (Aido Humanoid) and outdoor patrol (Aido Rover) are **scenario ranges, not forecasts** — pre-commercial / definitionally narrow; published totals disagree the most here.
- Full ranges, both methods, reconciliation, and sources are in `reports/week04/market_sizing_workbook.xlsx`; sensitivity in the `tornado_*.png` files; memo in `reports/week04/methodology_memo.pdf`.